In [ ]:
# stdlib pathlib (filesystem paths)
from pathlib import Path

# src/utils/pathing.py
from src.utils.pathing import ensure_repo_root_on_sys_path  # src/utils/pathing.py

ensure_repo_root_on_sys_path(Path.cwd())

# 15. Supervised Learning: Decision Trees

## Algorithm Category
**Type**: Supervised Learning - Classification/Regression  
**Complexity**: Medium  
**Use Case**: Tree-based decision making with interpretable rules

## Learning Objectives

By the end of this notebook, you will be able to:
- Understand how decision trees make decisions through recursive splitting
- Implement decision trees for both classification and regression
- Understand entropy, Gini impurity, and information gain
- Visualize decision trees and interpret their structure
- Tune hyperparameters to prevent overfitting
- Apply decision trees to real-world problems

## Historical Context

Decision trees have roots in decision theory and were formalized in the 1960s. Key developments include:
- ID3 algorithm (Quinlan, 1986) - used information gain
- C4.5 algorithm (Quinlan, 1993) - improved handling of continuous features
- CART algorithm (Breiman et al., 1984) - Classification and Regression Trees

**Key Papers/References:**
- Breiman, L., et al. (1984). "Classification and Regression Trees"
- Quinlan, J.R. (1986). "Induction of Decision Trees"
- Quinlan, J.R. (1993). "C4.5: Programs for Machine Learning"

## When to Use Decision Trees

Decision trees are appropriate when:
- Interpretability is crucial (easy to visualize and explain)
- You need to understand feature importance
- Data has non-linear relationships
- You want a baseline before using ensemble methods
- Working with mixed data types (categorical and numerical)

## Theory & Mechanics

### Mathematical Foundation

Decision trees recursively partition the feature space by asking yes/no questions about feature values.

**Entropy (for classification):**
$$H(S) = -\sum_{i=1}^{c} p_i \log_2(p_i)$$

**Gini Impurity:**
$$Gini(S) = 1 - \sum_{i=1}^{c} p_i^2$$

**Information Gain:**
$$IG(S, A) = H(S) - \sum_{v \in Values(A)} \frac{|S_v|}{|S|} H(S_v)$$

**Variance Reduction (for regression):**
$$\text{Var}(S) = \frac{1}{n}\sum_{i=1}^{n}(y_i - \bar{y})^2$$

### How It Works

1. **Start**: Begin with all training data at root node
2. **Split**: Find best feature and threshold that maximizes information gain (or minimizes impurity)
3. **Recurse**: Repeat for each child node until stopping criterion met
4. **Leaf**: Assign class (majority vote) or value (mean) to leaf nodes
5. **Prediction**: Traverse tree from root to leaf based on feature values

### Key Hyperparameters

- **max_depth**: Maximum depth of tree (prevents overfitting)
- **min_samples_split**: Minimum samples required to split a node
- **min_samples_leaf**: Minimum samples required in a leaf node
- **max_features**: Number of features to consider for best split
- **criterion**: Splitting criterion ('gini', 'entropy' for classification; 'mse', 'mae' for regression)

### Limitations

- Prone to overfitting (high variance)
- Sensitive to small changes in data
- Can create biased trees if classes are imbalanced
- May not capture linear relationships efficiently


## Implementation

Let's implement decision trees for both classification and regression.

**Implementation Steps:**
1. **Import Libraries**: Load necessary tools and datasets
2. **Load Data**: Get classification and regression datasets
3. **Train Models**: Fit decision trees for both tasks
4. **Visualize Trees**: See how the tree makes decisions
5. **Evaluate**: Calculate performance metrics
6. **Tune Hyperparameters**: Prevent overfitting
7. **Interpret**: Understand feature importance and decision paths


In [ ]:
# ============================================
# IMPORTING LIBRARIES: Setting Up Our Tools
# ============================================

# Core data science libraries
import numpy as np  # NumPy: Numerical computing (arrays, math operations)
import pandas as pd  # Pandas: Data manipulation (DataFrames, data analysis)
import matplotlib.pyplot as plt  # Matplotlib: Plotting and visualization

# Scikit-learn: Machine learning library
from sklearn.datasets import load_iris, load_diabetes  # Classification and regression datasets
from sklearn.tree import (
    DecisionTreeClassifier,  # Decision tree for classification
    DecisionTreeRegressor,  # Decision tree for regression
    plot_tree,  # Visualize decision tree structure
    export_text  # Export tree as text (readable rules)
)
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV  # Model selection tools
from sklearn.metrics import accuracy_score, classification_report, mean_squared_error  # Evaluation metrics

# ============================================
# IMPORTING OUR HELPER FUNCTIONS
# ============================================

# Our custom utility functions (organized in src/ directory)
from src.models.supervised import split_data, evaluate_classifier, evaluate_regressor  # Supervised learning utilities
from src.models.classification import calculate_classification_metrics, plot_confusion_matrix  # Classification utilities
from src.utils.benchmarking import benchmark_model_training  # Measure training time
from src.utils.traceability import (
    extract_feature_importance_trace,  # Extract feature importance
    trace_decision_path,  # Trace how a sample goes through the tree
    save_traceability_data  # Save model information
)
from src.utils.validation import validate_model_output, check_cross_validation_stability  # Model validation

print("Libraries imported successfully!")  # Confirm all imports worked


In [ ]:
# ============================================
# LOADING THE DATASET: Iris Classification
# ============================================

# load_iris() loads the Iris flower dataset from scikit-learn
# This is a multiclass classification problem: predict flower species from measurements
# No download needed - it's built into scikit-learn
iris = load_iris()  # Returns a Bunch object with data, target, feature_names

# X = Features (inputs): Flower measurements
# iris.data contains feature values (150 samples × 4 features)
# We convert to DataFrame for easier manipulation
# columns=iris.feature_names gives meaningful column names
X = pd.DataFrame(iris.data, columns=iris.feature_names)
# Features: sepal length, sepal width, petal length, petal width (4 measurements)

# y = Target (output): Flower species (what we want to predict)
# iris.target contains class labels (0, 1, or 2 for each sample)
# We convert to Series and give it a descriptive name
y = pd.Series(iris.target, name='Species')
# 0 = setosa, 1 = versicolor, 2 = virginica

# ============================================
# EXPLORING THE DATASET: Understanding Our Data
# ============================================

# .shape returns (rows, columns) - dimensions of the dataset
print(f"Dataset Shape: {X.shape}")  # Output: (150, 4) - 150 flowers, 4 features

# Display class names (what the target values represent)
print(f"Classes: {iris.target_names.tolist()}")  # Output: ['setosa', 'versicolor', 'virginica']

# .value_counts() counts how many samples belong to each class
print(f"Class distribution:\n{y.value_counts()}")  # Shows: 50 samples per class (balanced dataset)

# ============================================
# TRAIN/TEST SPLIT: Separating Data
# ============================================

# split_data() randomly splits data into training (80%) and test (20%) sets
# test_size=0.2 means 20% for testing, 80% for training
# random_state=42 ensures same split every time (reproducibility)
X_train, X_test, y_train, y_test = split_data(X, y, test_size=0.2, random_state=42)
# X_train: 120 samples for training
# X_test: 30 samples for testing
# y_train: Labels for training samples
# y_test: Labels for test samples (ground truth)


In [ ]:
# ============================================
# MODEL CREATION: Decision Tree Classifier
# ============================================

# Create a DecisionTreeClassifier model object
# This model will learn a tree structure by asking yes/no questions about features
# max_depth=3: Limit tree depth to 3 levels (prevents overfitting)
#   Without this, tree might grow too deep and memorize training data
# random_state=42: Ensures reproducible results
model = DecisionTreeClassifier(max_depth=3, random_state=42)

# ============================================
# MODEL TRAINING: Learning from Data
# ============================================

# .fit() trains the model on training data
# The model learns:
# 1. Which features to split on (e.g., "petal length > 2.5?")
# 2. What thresholds to use for splits
# 3. When to stop splitting (create leaf nodes)
# Uses information gain or Gini impurity to choose best splits
model.fit(X_train, y_train)  # Train the model

print("Model trained successfully!")  # Confirm training completed

# ============================================
# TREE STRUCTURE: Understanding the Tree
# ============================================

# .get_depth() returns the actual depth of the trained tree
# May be less than max_depth if early stopping occurred
print(f"Tree depth: {model.get_depth()}")  # Actual depth (≤ max_depth)

# .get_n_leaves() returns number of leaf nodes (terminal nodes)
# Leaf nodes make final predictions (no more splits)
print(f"Number of leaves: {model.get_n_leaves()}")  # Number of decision endpoints

# ============================================
# MAKING PREDICTIONS: Using the Trained Tree
# ============================================

# .predict() uses the trained tree to make predictions
# For each sample, it:
# 1. Starts at root node
# 2. Follows path based on feature values (e.g., "if petal length > 2.5, go right")
# 3. Reaches a leaf node
# 4. Returns the class assigned to that leaf
y_pred = model.predict(X_test)  # Predictions: array of class labels (0, 1, or 2)

# Calculate accuracy: (correct predictions) / (total predictions)
accuracy = accuracy_score(y_test, y_pred)  # Compare predictions to true labels
print(f"\nTest Accuracy: {accuracy:.3f}")  # Display accuracy (e.g., 0.967 = 96.7% correct)


In [ ]:
# ============================================
# VISUALIZING THE DECISION TREE: Understanding the Model
# ============================================

# plot_tree() creates a visual representation of the decision tree
# This helps understand how the model makes decisions
# You can trace a path from root to leaf to see the decision process

# Create large figure (tree can be wide and tall)
plt.figure(figsize=(20, 10))  # Width=20 inches, height=10 inches
# Large figure needed because trees can have many nodes

# plot_tree() visualizes the tree structure
plot_tree(
    model,  # The trained decision tree
    feature_names=iris.feature_names,  # Feature names for labels (e.g., "sepal length (cm)")
    class_names=iris.target_names,  # Class names for labels (e.g., "setosa", "versicolor", "virginica")
    filled=True,  # Fill nodes with colors (different colors for different classes)
    rounded=True  # Use rounded rectangles (more visually appealing)
)
# The tree shows:
#   - Root node at top (first decision)
#   - Internal nodes (intermediate decisions)
#   - Leaf nodes at bottom (final predictions)
#   - Each node shows: feature, threshold, samples, value, class

plt.title("Decision Tree Visualization")  # Chart title
plt.show()  # Display the tree

# Interpretation:
# - Start at root: First question (e.g., "Is petal length ≤ 2.45?")
# - Follow path: Answer yes → go left, answer no → go right
# - Reach leaf: Final prediction (e.g., "setosa")
# - Colors: Different colors for different classes
# - This visualization helps understand model decisions (interpretability!)


In [ ]:
# ============================================
# TREE AS TEXT: Reading Decision Rules
# ============================================

# export_text() converts the tree to human-readable text format
# This is useful for understanding the decision logic
# Each line shows a decision rule (if-then statement)

# export_text() creates a text representation of the tree
tree_rules = export_text(
    model,  # The trained decision tree
    feature_names=iris.feature_names  # Feature names for labels
)
# Returns: Multi-line string with tree structure

print("Decision Tree Rules:")
print(tree_rules)  # Display the tree as text

# The text format shows:
#   - Indentation indicates tree depth (more indented = deeper in tree)
#   - Lines like "|--- feature <= threshold" show decision rules
#   - Lines like "|   |   class: setosa" show leaf predictions
#   - "value = [a, b, c]" shows class distribution at that node
#     (e.g., [50, 0, 0] means 50 samples of class 0, 0 of class 1, 0 of class 2)

# Interpretation:
# - Read from top to bottom: decision path
# - Example: "if petal length <= 2.45 then setosa"
# - This text format is useful for:
#   - Understanding model logic
#   - Explaining decisions to non-technical audiences
#   - Debugging model behavior
#   - Creating if-then rules for other systems


## Validation & Testing

Let's validate our model and check for overfitting.


In [ ]:
# ============================================
# VALIDATION 1: Checking for Overfitting
# ============================================

# Overfitting = model performs well on training data but poorly on test data
# This happens when the tree memorizes training data instead of learning patterns
# We check by comparing training vs test accuracy

# Make predictions on training data (data the model saw during training)
train_pred = model.predict(X_train)  # Predictions on training set
train_accuracy = accuracy_score(y_train, train_pred)  # How well it did on training data

# Test accuracy (already calculated, but shown for comparison)
test_accuracy = accuracy_score(y_test, y_pred)  # How well it did on test data

print("Overfitting Check:")
print(f"  Training Accuracy: {train_accuracy:.3f}")  # Accuracy on data it trained on
print(f"  Test Accuracy: {test_accuracy:.3f}")  # Accuracy on unseen data
print(f"  Difference: {train_accuracy - test_accuracy:.3f}")  # Gap between them

# Interpretation:
# - Small difference (< 0.05) = good generalization (not overfitting)
# - Large difference (> 0.10) = overfitting (memorized training data)
# - If test > train, might be underfitting or lucky split

# ============================================
# ASSERTIONS: Automated Validation Checks
# ============================================

# validate_model_output() checks if predictions are valid
# task_type='classification' tells validator this is classification
validation_result = validate_model_output(y_pred, y_test.values, task_type='classification')
# Returns dictionary with validation results

# Check 1: Predictions must be valid
assert validation_result['valid'], "Invalid predictions!"
# If predictions are invalid (wrong shape, wrong values, etc.), stop execution

# Check 2: Test accuracy must be better than random guessing
# For 3-class classification, random = 1/3 ≈ 0.333
assert test_accuracy > 0.5, "Test accuracy should be better than random!"
# If accuracy ≤ 0.5, model is no better than guessing

print("\n✓ Basic validation checks passed")  # All checks passed!


In [ ]:
# ============================================
# VALIDATION 2: Cross-Validation
# ============================================

# Cross-validation splits data into k folds (groups)
# Trains on k-1 folds, tests on 1 fold
# Repeats k times (each fold used as test set once)
# More reliable than single train/test split (reduces variance)

# cross_val_score() performs k-fold cross-validation
# model: The decision tree to evaluate
# X, y: All data (will be split internally)
# cv=5: 5 folds (5 train/test splits)
# scoring='accuracy': Use accuracy as the evaluation metric
cv_scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
# Returns: array of 5 accuracy scores (one per fold)
# Example: [0.9667, 0.9667, 0.9333, 1.0, 0.9667]

# Calculate statistics across folds
cv_mean = cv_scores.mean()  # Average accuracy across all folds
cv_std = cv_scores.std()  # Standard deviation (measure of variability)

print(f"\nCross-Validation Results (5-fold):")
print(f"  Mean Accuracy: {cv_mean:.3f} (+/- {cv_std:.3f})")  # Average ± variability

# Interpretation:
# - CV accuracy is more reliable than single train/test split
# - Low std = consistent performance across folds (stable model)
# - High std = inconsistent performance (model sensitive to data split)
# - CV mean should be close to test accuracy (if not, model may be overfitting)

# Calculate statistics across folds
cv_mean = cv_scores.mean()  # Average accuracy across all folds
cv_std = cv_scores.std()  # Standard deviation (measure of variability)

print("Cross-Validation Results (5-fold):")
print(f"  Mean Accuracy: {cv_mean:.3f} (+/- {cv_std:.3f})")  # Average ± variability
print(f"  Individual fold scores: {cv_scores}")  # Accuracy for each of the 5 folds

# ============================================
# STABILITY CHECK: Is Performance Consistent?
# ============================================

# Check if cross-validation results are stable (low variation)
# Unstable results suggest model is sensitive to data split
# Stable results suggest model is robust

# check_cross_validation_stability() calculates coefficient of variation
# threshold=0.1 means "variation should be less than 10% of mean"
stability = check_cross_validation_stability(cv_scores, threshold=0.1)
# Returns dictionary with stability analysis

print(f"  Is Stable: {stability['is_stable']}")  # True if variation < threshold

# ============================================
# ASSERTIONS: Cross-Validation Checks
# ============================================

# Check: CV accuracy must be better than random
assert cv_mean > 0.5, "CV accuracy should be better than random!"
# Random guessing = 1/3 ≈ 0.333 for 3 classes

# Check: Results must be stable
assert stability['is_stable'], "Cross-validation results are unstable!"
# Unstable results suggest model is unreliable

print("\n✓ Cross-validation checks passed")  # All checks passed!


## Performance Benchmarking

Let's benchmark performance and compare different tree depths.


In [ ]:
# ============================================
# COMPARING TREE DEPTHS: Finding Optimal Complexity
# ============================================

# max_depth controls how deep the tree can grow
# Shallow trees (low depth) = simple, may underfit
# Deep trees (high depth) = complex, may overfit
# We'll test different depths to find the sweet spot

# Test depths from 1 to 10
depths = range(1, 11)  # [1, 2, 3, ..., 10]
train_scores = []  # Store training accuracy for each depth
test_scores = []  # Store test accuracy for each depth

# Loop through each depth value
for depth in depths:
    # Create a new tree with this max_depth
    dt = DecisionTreeClassifier(max_depth=depth, random_state=42)
    
    # Train the tree
    dt.fit(X_train, y_train)
    
    # Calculate training accuracy (how well it fits training data)
    train_scores.append(accuracy_score(y_train, dt.predict(X_train)))
    
    # Calculate test accuracy (how well it generalizes)
    test_scores.append(accuracy_score(y_test, dt.predict(X_test)))

# ============================================
# VISUALIZATION: Training vs Test Performance
# ============================================

# Create line plot comparing training and test accuracy
plt.figure(figsize=(10, 6))  # Figure size: 10×6 inches

# Plot training accuracy
# 'o-' means circle markers connected by lines
plt.plot(depths, train_scores, 'o-', label='Training Accuracy')

# Plot test accuracy
# 's-' means square markers connected by lines
plt.plot(depths, test_scores, 's-', label='Test Accuracy')

# Label axes
plt.xlabel('Max Depth')  # X-axis: tree depth
plt.ylabel('Accuracy')  # Y-axis: accuracy score
plt.title('Decision Tree Performance vs Depth')  # Chart title
plt.legend()  # Show legend (which line is which)
plt.grid(True, alpha=0.3)  # Add grid (alpha=0.3 makes it semi-transparent)

# Adjust layout
plt.tight_layout()
plt.show()  # Display the plot

# Interpretation:
# - Training accuracy increases with depth (tree fits training data better)
# - Test accuracy increases then decreases (optimal depth in middle)
# - Large gap between train and test = overfitting
# - Optimal depth = where test accuracy is highest

print("Optimal depth appears around max_depth=3-4")  # Based on the plot


## Traceability

Let's extract feature importance and trace decision paths.


In [ ]:
# ============================================
# FEATURE IMPORTANCE: Which Features Matter Most?
# ============================================

# In decision trees, feature importance = how much each feature reduces impurity
# Higher importance = feature is used more often and/or in more important splits
# extract_feature_importance_trace() extracts and organizes this information

feature_importance = extract_feature_importance_trace(
    model,  # The trained decision tree (has .feature_importances_ attribute)
    feature_names=X.columns.tolist()  # List of feature names
)
# Returns DataFrame with features sorted by importance (highest first)

print("Feature Importance:")
print(feature_importance)  # Display the importance values

# ============================================
# VISUALIZING FEATURE IMPORTANCE
# ============================================

# Create horizontal bar chart showing feature importance
plt.figure(figsize=(10, 6))  # Figure size: 10×6 inches

# Create horizontal bar chart
# range(len(feature_importance)): Y-axis positions (0, 1, 2, ..., n-1)
# feature_importance['importance']: Bar lengths (importance values)
# align='center': Center bars on Y-axis positions
plt.barh(range(len(feature_importance)), feature_importance['importance'], align='center')

# Set Y-axis labels to feature names
plt.yticks(range(len(feature_importance)), feature_importance['feature'])

# Label axes
plt.xlabel('Importance')  # X-axis: importance value (0 to 1)
plt.title('Feature Importance (Decision Tree)')  # Chart title

# Invert Y-axis so most important feature is at top
plt.gca().invert_yaxis()  # gca() = get current axes

# Adjust layout
plt.tight_layout()
plt.show()  # Display the plot

# Interpretation:
# - Longer bars = more important features
# - Features at top have largest impact on predictions
# - Sum of all importances = 1.0
# - Features with importance ≈ 0 are not used by the tree


In [ ]:
# ============================================
# TRACING DECISION PATH: How a Sample is Classified
# ============================================

# trace_decision_path() shows exactly how a sample goes through the tree
# This is useful for understanding why a specific prediction was made
# Helps with interpretability and debugging

# Select a sample from test set to trace
sample_idx = 0  # First sample in test set
sample = X_test.iloc[sample_idx:sample_idx+1].values[0]
# X_test.iloc[sample_idx:sample_idx+1] gets one row as DataFrame
# .values[0] converts to NumPy array (required by trace function)

# trace_decision_path() follows the sample through the tree
path_info = trace_decision_path(
    model,  # The trained decision tree
    sample,  # The sample to trace (feature values)
    feature_names=X.columns.tolist()  # Feature names (for readability)
)
# Returns dictionary with:
# - path: List of decision steps (which features were checked)
# - prediction: Final predicted class

# ============================================
# DISPLAYING THE DECISION PATH
# ============================================

# Show the sample's feature values
# dict(zip()) pairs feature names with their values
print(f"Sample: {dict(zip(X.columns, sample))}")
# Example: {'sepal length (cm)': 5.8, 'sepal width (cm)': 2.7, ...}

print(f"\nDecision Path:")
# Loop through each decision step in the path
for step in path_info['path']:
    # step['feature']: Which feature was checked (e.g., "petal length (cm)")
    # step['threshold']: What value was compared against (e.g., 2.45)
    # step['value']: The sample's actual value for this feature (e.g., 1.4)
    print(f"  {step['feature']} <= {step['threshold']:.3f} (value: {step['value']:.3f})")
    # Example: "petal length (cm) <= 2.450 (value: 1.400)"
    # This means: "Is petal length <= 2.45? Yes (because 1.4 <= 2.45), so go left"

# Show final prediction
print(f"\nPrediction: {iris.target_names[int(path_info['prediction'])]}")
# iris.target_names[int(...)] converts class number to name (e.g., 0 → "setosa")

# Show actual label (ground truth) for comparison
print(f"Actual: {iris.target_names[y_test.iloc[sample_idx]]}")

# Interpretation:
# - Shows exactly which features were used to make the prediction
# - Each step shows the condition and whether it was true/false
# - Final step shows the predicted class
# - Compare prediction vs actual to see if it was correct


## Regression Example

Let's also apply decision trees to regression.


In [ ]:
# Regression Example: Diabetes dataset
diabetes = load_diabetes()
X_reg = pd.DataFrame(diabetes.data, columns=diabetes.feature_names)
y_reg = pd.Series(diabetes.target, name='Target')

X_reg_train, X_reg_test, y_reg_train, y_reg_test = split_data(X_reg, y_reg, test_size=0.2, random_state=42)

# Train Decision Tree Regressor
dt_reg = DecisionTreeRegressor(max_depth=4, random_state=42)
dt_reg.fit(X_reg_train, y_reg_train)

y_reg_pred = dt_reg.predict(X_reg_test)
rmse = np.sqrt(mean_squared_error(y_reg_test, y_reg_pred))

print("Decision Tree Regression:")
print(f"  RMSE: {rmse:.3f}")
print(f"  Tree depth: {dt_reg.get_depth()}")
print(f"  Number of leaves: {dt_reg.get_n_leaves()}")


## Real-World Application

Let's tune hyperparameters using GridSearchCV.


In [ ]:
# Hyperparameter tuning
param_grid = {
    'max_depth': [3, 5, 7, 10],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

grid_search = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)
grid_search.fit(X_train, y_train)

print("Best Hyperparameters:")
print(grid_search.best_params_)
print(f"\nBest CV Accuracy: {grid_search.best_score_:.3f}")

# Train with best parameters
best_model = grid_search.best_estimator_
best_pred = best_model.predict(X_test)
best_accuracy = accuracy_score(y_test, best_pred)
print(f"Test Accuracy with Best Model: {best_accuracy:.3f}")


## Summary & Key Takeaways

### Key Concepts Learned

1. **Decision Tree Basics**
   - Recursive binary splitting based on feature values
   - Uses entropy/Gini for classification, variance for regression
   - Creates interpretable decision rules

2. **Splitting Criteria**
   - Information Gain: Maximize reduction in entropy
   - Gini Impurity: Measure of node impurity
   - Variance Reduction: For regression tasks

3. **Overfitting Prevention**
   - Limit tree depth (max_depth)
   - Require minimum samples to split (min_samples_split)
   - Require minimum samples in leaves (min_samples_leaf)

4. **Best Practices**
   - Start with small max_depth and increase gradually
   - Use cross-validation to find optimal hyperparameters
   - Visualize trees to understand model decisions
   - Use as baseline before ensemble methods

### When to Use Decision Trees

✅ **Good for:**
- Interpretability is important
- Non-linear relationships
- Mixed data types
- Feature importance analysis
- Baseline for ensemble methods

❌ **Not ideal for:**
- High accuracy requirements (use ensembles)
- Very large datasets (computational cost)
- Linear relationships (use linear models)
- Real-time predictions (can be slow)

### Next Steps

- Try **Random Forest** (ensemble of trees) for better accuracy
- Explore **Gradient Boosting** for sequential improvement
- Consider **XGBoost** for optimized tree boosting
- Use **Pruning** techniques to reduce overfitting
